In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [10]:
diabetes_dataset = pd.read_csv("./diabetes_experiment_dataset.csv")
diabetes_dataset.corr()["Outcome"].sort_values(ascending=False)

Outcome                     1.000000
Glucose                     0.466581
BMI                         0.292695
Age                         0.238356
Pregnancies                 0.221898
DiabetesPedigreeFunction    0.173844
Insulin                     0.130548
SkinThickness               0.074752
BloodPressure               0.065068
Name: Outcome, dtype: float64

In [12]:
diabetes_label = diabetes_dataset["Outcome"]
diabetes_dataset.drop(columns=["Outcome"], inplace=True)

In [13]:
diabetes_dataset

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [14]:
diabetes_label

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64

In [35]:
X_train, X_test, y_train, y_test = train_test_split(diabetes_dataset, diabetes_label, test_size=0.2, random_state=42, stratify=diabetes_label)


scale_features = ColumnTransformer(transformers=[
    ("scaling", StandardScaler(), X_train.columns)
])

randomForestModel = RandomForestClassifier(random_state=42)
xgboostModel = XGBClassifier()
kneighboursModel = KNeighborsClassifier()
svcModel = SVC(kernel="rbf")
logisticModel = LogisticRegression(max_iter=5000)

model_pipeline = Pipeline(steps=[
    ("scale", scale_features),
    ("model", logisticModel)
])

my_folds = StratifiedKFold(
    n_splits=5, 
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    scoring="recall",
    cv=my_folds,
    n_jobs=-1,
)

print(scores.mean())
print(scores.std())

0.5841638981173866
0.02993312667747729


In [ ]:
param_randomforest_grid = [
    {
        'model': [RandomForestClassifier(random_state=42, class_weight="balanced")],
        'model__n_estimators': [100, 200, 500],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', None]
    },
]

param_xgbclassifier_grid = [
 {
        'model': [XGBClassifier()],
        'model__n_estimators': [100, 200, 500],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__subsample': [0.7, 0.8, 1],
        'model__colsample_bytree': [0.7, 0.8, 1]
    },
]

param_knearest_grid = [
    {
        'model': [KNeighborsClassifier()],
        'model__n_neighbors': [3, 5, 7, 9],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },

]

param_svc_grid = [

    {
        'model': [SVC(kernel='rbf')],
        'model__C': [0.1, 1, 10, 100],
        'model__gamma': ['scale', 'auto', 0.01, 0.1, 1]
    },
]

param_decision_grid = [
    {
        'model': [DecisionTreeClassifier(random_state=42)],
        'model__max_depth': [None, 5, 10, 20],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', None]
    },
]

param_logisticReg_grid = [
    {
        'model': [LogisticRegression(max_iter=5000)],
        'model__C': [0.01, 0.1, 1, 10],
        'model__penalty': ['l2'],
        'model__solver': ['lbfgs', 'saga']
    }

]


grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_randomforest_grid,
    scoring="recall",
    n_jobs=-1, 
    cv=my_folds,
    return_train_score=True
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
{'model': RandomForestClassifier(random_state=42), 'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 1, 'model__min_samples_split': 10, 'model__n_estimators': 500}
{'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...), 'model__colsample_bytree': 1, 'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 500, 'model__subsample': 0.7}
0.6121816168327796




